# NLP Overview — Exercises

Companion to the note [NLP Overview](NLP%20Overview.md).

Practise the classic NLP pipeline end to end: normalisation and tokenisation (including BPE), n-gram language models and perplexity, the task zoo, and the evaluation metrics from the note (F1, BLEU, ROUGE, WER), first by hand and then from scratch in NumPy / plain Python.

**18 exercises** · 🧠 Concept ×5 · ✍️ By hand ×6 · 💻 Code ×7

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

import re
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
rng = np.random.default_rng(0)

# Tiny bigram-LM corpus used in Parts B and C
lm_corpus = ["<s> I like cats </s>", "<s> I like dogs </s>", "<s> you like cats </s>"]

## Part A · Concepts

### Exercise 1 · Normalisation choices matter
*🧠 Concept · ★☆☆*

The note's pipeline starts with **normalisation → tokenisation**. For a *sentiment classifier* on product reviews, discuss
whether each step helps or hurts:

1. lower-casing, 2. removing stop-words (including "not", "no"), 3. stemming ("running" → "run"),
4. stripping punctuation and emoji, 5. replacing numbers by a `<NUM>` token.

Would your answers change for a *named entity recogniser*?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Ask for each step: which information does it destroy, and does the task need that information?

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. **Lower-casing** shrinks the vocabulary and usually helps sentiment; it hurts NER, where capitalisation ("Apple" vs "apple") is a strong cue.
2. **Stop-word removal** is dangerous for sentiment: the standard lists contain *not/no/never*, so "not good" becomes "good". For topic classification it is mostly harmless.
3. **Stemming** merges inflections (fewer sparse features), but can conflate meanings ("university"/"universe" → "univers"). Modern subword tokenisers make it unnecessary.
4. **Punctuation and emoji** carry sentiment ("!!!", ":(", 😍); keep them or map them to tokens.
5. **`<NUM>`** is fine for sentiment (the exact number rarely matters) but loses information for NER/QA (dates, prices).

Rule of thumb: the stronger the downstream model (fine-tuned transformers), the *less* normalisation you do: the subword tokeniser handles raw text.

</details>

### Exercise 2 · Which head for which task?
*🧠 Concept · ★☆☆*

For each task in the note's table say whether the output is (a) one label per **sequence**, (b) one label per **token**,
(c) a **span** of the input, or (d) a **generated sequence**, and name a matching transformer architecture
(encoder-only, decoder-only, encoder–decoder):

sentiment · NER · POS tagging · machine translation · summarisation · extractive QA · semantic search · language modelling

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Encoder-only models (BERT) are good at *understanding* (labels, spans, embeddings); generation needs a decoder.

</details>

<details>
<summary><b>✅ Solution</b></summary>

| Task | Output | Typical architecture |
|---|---|---|
| Sentiment | (a) sequence label | encoder-only ([CLS] head) |
| NER, POS | (b) token labels (BIO tags) | encoder-only + token-classification head (historically CRF / BiLSTM-CRF) |
| Translation, summarisation | (d) generated sequence | encoder–decoder (T5, BART), or decoder-only LLM |
| Extractive QA | (c) start/end span | encoder-only with two pointer heads |
| Semantic search | a vector per text (no label) | encoder-only bi-encoder, see [[Text Representations]] |
| Language modelling | (d) next-token distribution | decoder-only ([[LLMs Overview]]) |

</details>

### Exercise 3 · Why attention rescued seq2seq
*🧠 Concept · ★★☆*

The note's history line goes "seq2seq + attention → Transformer". Explain the **bottleneck** of a plain RNN encoder–decoder,
how attention removes it, and what the Transformer then removed in turn (see [[RNN and LSTM]] and [[Transformers]]).

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

In a plain seq2seq model the whole source sentence must fit into one fixed-size vector.

</details>

<details>
<summary><b>✅ Solution</b></summary>

A plain RNN seq2seq model compresses the entire source sentence into the final hidden state $h_T$; long sentences lose
information and BLEU drops sharply with length. **Attention** lets each decoder step compute a weighted average
$c_t = \sum_s \alpha_{t,s} h_s$ over *all* encoder states, with weights from a similarity between decoder state and $h_s$,
so information no longer has to pass through one vector (and gradients get short paths).
The **Transformer** then dropped the recurrence entirely: self-attention relates all positions in $O(1)$ sequential steps,
which parallelises over the sequence on GPUs and makes pre-training at scale (BERT, GPT) feasible.

</details>

### Exercise 4 · Perplexity across tokenisers
*🧠 Concept · ★★★*

Model A (word-level vocabulary) reports a test perplexity of 80; model B (BPE vocabulary) reports 12 on the *same text*.
Can you conclude that B is the better language model? What quantity *would* be comparable?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Perplexity is $\exp$ of the average loss *per token*. Do both models have the same number of tokens for the text?

</details>

<details>
<summary><b>✅ Solution</b></summary>

No. Perplexity $= \exp\big(-\frac1T\sum_t \log p(x_t\mid x_{<t})\big)$ is normalised **per token**, and BPE splits the text
into more (shorter, easier) tokens, so the per-token number is not comparable. What is comparable is the **total**
log-likelihood of the text, normalised by a tokeniser-independent unit: bits per character / per byte, or word-level
perplexity $\exp(-\frac{1}{W}\sum_t \log p)$ where $W$ is the number of *words*. Example: if B uses 1.8 tokens per word,
its word-level perplexity is $12^{1.8} \approx 88$, worse than A's 80.

</details>

## Part B · By hand

### Exercise 5 · Bigram LM by maximum likelihood
*✍️ By hand · ★☆☆*

Using `lm_corpus` (three sentences: *I like cats*, *I like dogs*, *you like cats*, each wrapped in `<s> … </s>`),
estimate bigram probabilities by MLE, $P(w_i \mid w_{i-1}) = \frac{c(w_{i-1}, w_i)}{c(w_{i-1})}$, and compute
$P(\text{<s> you like dogs </s>})$.

In [ ]:
p_sentence = None  # probability of '<s> you like dogs </s>'

check('P(<s> you like dogs </s>)', p_sentence, 1/9)

<details>
<summary><b>💡 Hint 1</b></summary>

You need $P(\text{you}\mid\text{<s>})$, $P(\text{like}\mid\text{you})$, $P(\text{dogs}\mid\text{like})$ and $P(\text{</s>}\mid\text{dogs})$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

"like" occurs 3 times, followed by "dogs" once.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$P(\text{you}\mid\text{<s>}) = \tfrac13$, $P(\text{like}\mid\text{you}) = \tfrac11$, $P(\text{dogs}\mid\text{like}) = \tfrac13$,
$P(\text{</s>}\mid\text{dogs}) = 1$. Product: $\tfrac13\cdot 1\cdot\tfrac13\cdot 1 = \tfrac19 \approx 0.111$.
Note that the sentence never appeared in training but still gets non-zero probability: the bigram model generalises by
recombining seen pairs.

```python
p_sentence = 1/9
```

</details>

### Exercise 6 · Add-one (Laplace) smoothing
*✍️ By hand · ★★☆*

The MLE gives $P(\text{you}\mid\text{I}) = 0$, so any sentence containing "I you" has probability 0. With add-one smoothing
$P(w_i\mid w_{i-1}) = \frac{c(w_{i-1},w_i) + 1}{c(w_{i-1}) + V}$, where $V$ is the number of possible *next* tokens
(all types except `<s>`: I, like, cats, dogs, you, `</s>`).

Compute (a) $P(\text{dogs}\mid\text{like})$ and (b) $P(\text{you}\mid\text{I})$ after smoothing.

In [ ]:
p_dogs_like = None
p_you_I = None

check('P(dogs|like) smoothed', p_dogs_like, 2/9)
check('P(you|I) smoothed', p_you_I, 1/8)

<details>
<summary><b>💡 Hint</b></summary>

$V = 6$. "like" occurs 3 times, "I" twice.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $\frac{1+1}{3+6} = \frac29 \approx 0.222$ (down from the MLE $\frac13$). (b) $\frac{0+1}{2+6} = \frac18 = 0.125$ (up from 0).
Smoothing moves probability mass from seen to unseen events. Add-one moves *too much* for large vocabularies, which is why
add-$k$ ($k<1$), Kneser–Ney or back-off are used in practice.

```python
p_dogs_like = 2/9
p_you_I = 1/8
```

</details>

### Exercise 7 · Perplexity by hand
*✍️ By hand · ★☆☆*

A language model assigns the four tokens of a test sentence the probabilities $0.5, 0.25, 0.125, 0.5$.
Compute the perplexity $PP = \big(\prod_t p_t\big)^{-1/T}$ and the average cross-entropy in **bits** per token.

In [ ]:
pp = None
bits_per_token = None

_p = np.array([0.5, 0.25, 0.125, 0.5])
check('perplexity', pp, np.exp(-np.mean(np.log(_p))))
check('bits per token', bits_per_token, -np.mean(np.log2(_p)))

<details>
<summary><b>💡 Hint</b></summary>

Work in $\log_2$: the probabilities are $2^{-1}, 2^{-2}, 2^{-3}, 2^{-1}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\log_2$ probabilities: $-1,-2,-3,-1$, mean $-1.75$, so the cross-entropy is $H = 1.75$ bits/token and
$PP = 2^{H} = 2^{1.75} \approx 3.364$. Interpretation: the model is as uncertain as a uniform choice among about 3.4 tokens.

```python
pp = 2 ** 1.75
bits_per_token = 1.75
```

</details>

### Exercise 8 · Entity-level precision, recall, F1
*✍️ By hand · ★☆☆*

An NER system predicts 4 entities in a document that has 5 gold entities; 3 predictions match a gold entity exactly
(same span and type). Compute precision, recall and $F_1$.

In [ ]:
prec = None
rec = None
f1 = None

check('precision', prec, 3/4)
check('recall', rec, 3/5)
check('F1', f1, 2*3/(4+5))

<details>
<summary><b>💡 Hint</b></summary>

$F_1 = \frac{2PR}{P+R}$, equivalently $\frac{2\,TP}{\#\text{pred} + \#\text{gold}}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$P = 3/4 = 0.75$, $R = 3/5 = 0.6$, $F_1 = \frac{2\cdot0.75\cdot0.6}{1.35} = \frac{6}{9} \approx 0.667$.

```python
prec, rec = 0.75, 0.6
f1 = 2 * prec * rec / (prec + rec)
```

</details>

### Exercise 9 · BLEU-2 by hand
*✍️ By hand · ★★☆*

Reference: *the cat sat on the mat*. Candidate: *the the cat sat on the mat*.

1. Compute the **clipped** unigram precision $p_1$ and bigram precision $p_2$ (clip each n-gram count at its count in the reference).
2. Compute the brevity penalty $BP = \min(1, e^{1 - r/c})$ with $r, c$ the reference and candidate lengths.
3. Compute $\text{BLEU-2} = BP\cdot\sqrt{p_1 p_2}$.

In [ ]:
p1 = None
p2 = None
bleu2 = None

check('p1', p1, 6/7)
check('p2', p2, 5/6)
check('BLEU-2', bleu2, np.sqrt(6/7*5/6))

<details>
<summary><b>💡 Hint 1</b></summary>

The candidate has 7 tokens; 'the' appears 3 times in it but only twice in the reference.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Candidate bigrams: the-the, the-cat, cat-sat, sat-on, on-the, the-mat.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Unigrams: the (3 → clipped to 2), cat, sat, on, mat → $p_1 = 6/7 \approx 0.857$. Bigrams: 5 of the 6 occur in the reference
(only *the the* does not) → $p_2 = 5/6$.
2. $c = 7 > r = 6$, so $BP = 1$.
3. $\text{BLEU-2} = \sqrt{\tfrac67\cdot\tfrac56} = \sqrt{5/7} \approx 0.845$.

Clipping is what stops "the the the the" from getting $p_1 = 1$; the brevity penalty stops very short candidates from gaming precision.

```python
p1, p2 = 6/7, 5/6
bleu2 = 1.0 * np.sqrt(p1 * p2)
```

</details>

### Exercise 10 · ROUGE-1 for summarisation
*✍️ By hand · ★☆☆*

Reference summary: *the cat was found under the bed* (7 tokens). System summary: *the cat was under the bed*.
Compute ROUGE-1 recall, precision and $F_1$ (clipped unigram overlap). Why is ROUGE traditionally **recall**-oriented
while BLEU is precision-oriented?

In [ ]:
rouge1_recall = None
rouge1_precision = None

check('ROUGE-1 recall', rouge1_recall, 6/7)
check('ROUGE-1 precision', rouge1_precision, 1.0)

<details>
<summary><b>💡 Hint</b></summary>

Overlap: the (2), cat, was, under, bed.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Overlap $= 6$. Recall $= 6/7 \approx 0.857$, precision $= 6/6 = 1$, $F_1 = \frac{2\cdot 6/7}{1 + 6/7} = 12/13 \approx 0.923$.
Summaries are judged on whether they **cover** the reference content (recall); translations are judged on whether what they
say is **correct** (precision) and length is controlled separately by the brevity penalty. Both are surface-overlap metrics,
which is why BERTScore, human evaluation and LLM-as-judge are listed alongside them in the note.

```python
rouge1_recall = 6/7
rouge1_precision = 1.0
```

</details>

## Part C · Code from scratch

### Exercise 11 · A regex tokeniser
*💻 Code · ★☆☆*

Implement `tokenize(text)` that lower-cases the text and returns all tokens of **two or more** word characters
(this is scikit-learn's default `token_pattern`, `\b\w\w+\b`). Compare with `CountVectorizer().build_analyzer()`.

In [ ]:
def tokenize(text):
    # TODO: lower-case, then re.findall with the right pattern
    return None

sample = "Don't panic! NLP in 2024 is fun, isn't it? A b c."
toks = tokenize(sample)
print(toks)

check('matches sklearn analyzer', toks == CountVectorizer().build_analyzer()(sample) if toks is not None else None, True)

<details>
<summary><b>💡 Hint</b></summary>

`re.findall(r"\b\w\w+\b", text.lower())`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Note what this simple tokeniser does: "Don't" becomes `['don']` (the "t" is dropped, being one character), and single-letter words
("A", "b") vanish. Such choices silently change the features, which is one motivation for learned subword tokenisers.

```python
def tokenize(text):
    return re.findall(r"\b\w\w+\b", text.lower())

sample = "Don't panic! NLP in 2024 is fun, isn't it? A b c."
toks = tokenize(sample)
print(toks)
```

</details>

### Exercise 12 · Byte-pair encoding: learn the merges
*💻 Code · ★★☆*

Implement BPE training (Sennrich et al. 2016). Represent each word as a tuple of characters plus an end-of-word marker
`'</w>'`, weighted by its frequency. Repeat `n_merges` times: count all adjacent symbol pairs (weighted), merge the most
frequent pair everywhere, record it. **Break ties by the lexicographically smallest pair** so the result is deterministic.

Run it on the classic corpus `{'low': 5, 'lower': 2, 'newest': 6, 'widest': 3}` and also write `apply_bpe(word, merges)`
that segments a new word by applying the merges in order. How is the unseen word "lowest" segmented?

In [ ]:
def learn_bpe(word_freqs, n_merges):
    # TODO: return a list of merged pairs, e.g. [('e', 's'), ...]
    return None

def apply_bpe(word, merges):
    # TODO: return a list of subword symbols
    return None

bpe_corpus = {'low': 5, 'lower': 2, 'newest': 6, 'widest': 3}
merges = learn_bpe(bpe_corpus, 10)
seg_lowest = apply_bpe('lowest', merges) if merges is not None else None
print(merges, seg_lowest)

check('first five merges', merges[:5] == [('e','s'), ('es','t'), ('est','</w>'), ('l','o'), ('lo','w')] if merges is not None else None, True)
check('lowest = low + est</w>', seg_lowest == ['low', 'est</w>'] if seg_lowest is not None else None, True)

<details>
<summary><b>💡 Hint 1</b></summary>

Count pairs with a `Counter`; `m = max(pairs.values()); best = min(p for p, c in pairs.items() if c == m)`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

To merge, scan each symbol tuple left to right; when `(sym[i], sym[i+1]) == best` emit `sym[i] + sym[i+1]` and skip two.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The first merges are `e s` (9 = 6 from *newest* + 3 from *widest*), `es t`, `est </w>`, `l o` (7), `lo w` (7), …
"lowest" never occurred, yet it is segmented into the meaningful pieces `low` + `est</w>`: BPE handles open vocabularies
with no `<UNK>` token, which is why GPT-style models use it ([[LLMs Overview]]). WordPiece picks merges by likelihood
gain instead of raw frequency; SentencePiece works on raw text including spaces.

```python
def _merge(sym, pair):
    out, i = [], 0
    while i < len(sym):
        if i < len(sym) - 1 and (sym[i], sym[i + 1]) == pair:
            out.append(sym[i] + sym[i + 1]); i += 2
        else:
            out.append(sym[i]); i += 1
    return tuple(out)

def learn_bpe(word_freqs, n_merges):
    vocab = {tuple(w) + ('</w>',): f for w, f in word_freqs.items()}
    merges = []
    for _ in range(n_merges):
        pairs = Counter()
        for sym, f in vocab.items():
            for a, b in zip(sym, sym[1:]):
                pairs[(a, b)] += f
        if not pairs:
            break
        m = max(pairs.values())
        best = min(p for p, c in pairs.items() if c == m)
        merges.append(best)
        vocab = {_merge(sym, best): f for sym, f in vocab.items()}
    return merges

def apply_bpe(word, merges):
    sym = tuple(word) + ('</w>',)
    for pair in merges:
        sym = _merge(sym, pair)
    return list(sym)

bpe_corpus = {'low': 5, 'lower': 2, 'newest': 6, 'widest': 3}
merges = learn_bpe(bpe_corpus, 10)
seg_lowest = apply_bpe('lowest', merges)
print(merges, seg_lowest)
```

</details>

### Exercise 13 · Bigram language model in code
*💻 Code · ★★☆*

Implement `train_bigram(sentences)` returning bigram and unigram (context) counts, and `sentence_prob(sent, k)` with
add-$k$ smoothing ($k=0$ is MLE; $V$ = number of types excluding `<s>`). Then compute the **perplexity** of the MLE model on
"<s> you like dogs </s>" (4 predicted tokens). Verify your Part B answers.

In [ ]:
def train_bigram(sentences):
    # TODO: return (bigram_counts, context_counts, vocab_size)
    return None

def sentence_prob(sent, model, k=0.0):
    # TODO: product of smoothed bigram probabilities
    return None

lm = train_bigram(lm_corpus)
p_mle = sentence_prob("<s> you like dogs </s>", lm, k=0) if lm is not None else None
p_add1 = sentence_prob("<s> you like dogs </s>", lm, k=1) if lm is not None else None
pp_mle = None  # perplexity of the MLE model on that sentence

check('MLE probability', p_mle, 1/9)
check('add-one probability', p_add1, (2/9)*(2/7)*(2/9)*(2/7))
check('perplexity', pp_mle, np.sqrt(3))

<details>
<summary><b>💡 Hint</b></summary>

Perplexity $= P(\text{sentence})^{-1/T}$ with $T=4$ predictions (`<s>` itself is not predicted).

</details>

<details>
<summary><b>✅ Solution</b></summary>

$P_{\text{MLE}} = 1/9$, so $PP = (1/9)^{-1/4} = \sqrt3 \approx 1.732$. With add-one smoothing each factor is
$\frac{c+1}{c(\text{ctx}) + 6}$: $\frac29\cdot\frac27\cdot\frac29\cdot\frac27 \approx 0.00403$, much smaller, because
mass was moved to the many unseen bigrams.

```python
def train_bigram(sentences):
    big, ctx, types = Counter(), Counter(), set()
    for s in sentences:
        toks = s.split()
        types.update(toks)
        for a, b in zip(toks, toks[1:]):
            big[(a, b)] += 1
            ctx[a] += 1
    return big, ctx, len(types - {'<s>'})

def sentence_prob(sent, model, k=0.0):
    big, ctx, V = model
    toks = sent.split()
    p = 1.0
    for a, b in zip(toks, toks[1:]):
        p *= (big[(a, b)] + k) / (ctx[a] + k * V)
    return p

lm = train_bigram(lm_corpus)
p_mle = sentence_prob("<s> you like dogs </s>", lm, k=0)
p_add1 = sentence_prob("<s> you like dogs </s>", lm, k=1)
pp_mle = p_mle ** (-1 / 4)
```

</details>

### Exercise 14 · Word error rate with edit distance
*💻 Code · ★★☆*

Implement `edit_distance(a, b)` (Levenshtein: insertions, deletions, substitutions cost 1) with dynamic programming over
two sequences (strings or token lists). Then compute the **word error rate**
$\text{WER} = \text{edit\_distance}(\text{ref words}, \text{hyp words}) / |\text{ref words}|$ for
ref = "the cat sat on the mat", hyp = "the cat sit on mat".

In [ ]:
def edit_distance(a, b):
    # TODO: DP table D[i][j] = distance between a[:i] and b[:j]
    return None

wer = None

check('kitten -> sitting', edit_distance('kitten', 'sitting'), 3)
check('flaw -> lawn', edit_distance('flaw', 'lawn'), 2)
check('WER', wer, 2/6)

<details>
<summary><b>💡 Hint</b></summary>

$D[i][j] = \min(D[i-1][j]+1,\ D[i][j-1]+1,\ D[i-1][j-1] + [a_i \ne b_j])$ with $D[i][0]=i$, $D[0][j]=j$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

One substitution (sat → sit) and one deletion (the) → distance 2, WER $= 2/6 \approx 0.333$. WER is the standard ASR metric;
the same DP with characters gives CER, and a variant (TER) is used for translation.

```python
def edit_distance(a, b):
    D = np.zeros((len(a) + 1, len(b) + 1), dtype=int)
    D[:, 0] = np.arange(len(a) + 1)
    D[0, :] = np.arange(len(b) + 1)
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            D[i, j] = min(D[i - 1, j] + 1, D[i, j - 1] + 1, D[i - 1, j - 1] + (a[i - 1] != b[j - 1]))
    return int(D[-1, -1])

ref, hyp = "the cat sat on the mat".split(), "the cat sit on mat".split()
wer = edit_distance(ref, hyp) / len(ref)
```

</details>

### Exercise 15 · Entity F1 from BIO tags
*💻 Code · ★★☆*

Token-classification models output BIO tags. Write `bio_to_spans(tags)` returning a set of `(type, start, end)` spans
(end exclusive) and `entity_f1(gold, pred)` (exact-match spans). Compute it for the sequences below, and compare with
plain token accuracy. Why do NER papers report entity-level F1?

In [ ]:
gold_tags = ['B-PER','I-PER','O','B-LOC','O','B-ORG','I-ORG','O','B-LOC']
pred_tags = ['B-PER','I-PER','O','B-LOC','O','B-ORG','O',    'O','B-PER']

def bio_to_spans(tags):
    # TODO
    return None

def entity_f1(gold, pred):
    # TODO: return (precision, recall, f1)
    return None

ent = entity_f1(gold_tags, pred_tags)
tok_acc = np.mean([g == p for g, p in zip(gold_tags, pred_tags)])
print(ent, tok_acc)

check('entity F1', None if ent is None else ent[2], 0.5)
check('spans of gold', None if bio_to_spans(gold_tags) is None else bio_to_spans(gold_tags) == {('PER',0,2),('LOC',3,4),('ORG',5,7),('LOC',8,9)}, True)

<details>
<summary><b>💡 Hint</b></summary>

Open a span at every `B-X`; extend it while the next tag is `I-X`; close it on anything else (`O`, another `B-`, or end).

</details>

<details>
<summary><b>✅ Solution</b></summary>

Gold spans: PER[0,2), LOC[3,4), ORG[5,7), LOC[8,9). Predicted: PER[0,2), LOC[3,4), ORG[5,6), PER[8,9).
Two exact matches → $P = R = F_1 = 0.5$, while token accuracy is $7/9 \approx 0.78$. Token accuracy is inflated by the many easy
`O` tokens and gives partial credit for truncated or mistyped entities; downstream users need whole, correctly typed entities.

```python
gold_tags = ['B-PER','I-PER','O','B-LOC','O','B-ORG','I-ORG','O','B-LOC']
pred_tags = ['B-PER','I-PER','O','B-LOC','O','B-ORG','O',    'O','B-PER']

def bio_to_spans(tags):
    spans, start, typ = set(), None, None
    for i, t in enumerate(list(tags) + ['O']):
        if start is not None and t != 'I-' + typ:
            spans.add((typ, start, i)); start = None
        if t.startswith('B-'):
            start, typ = i, t[2:]
    return spans

def entity_f1(gold, pred):
    g, p = bio_to_spans(gold), bio_to_spans(pred)
    tp = len(g & p)
    prec, rec = tp / len(p), tp / len(g)
    return prec, rec, 0.0 if tp == 0 else 2 * prec * rec / (prec + rec)

ent = entity_f1(gold_tags, pred_tags)
tok_acc = np.mean([g == p for g, p in zip(gold_tags, pred_tags)])
print(ent, tok_acc)
```

</details>

### Exercise 16 · Negation needs n-grams
*💻 Code · ★★★*

The tiny sentiment set below contains "good movie" (+), "not good" (−), "bad movie" (−) and "not bad" (+).

1. Prove on paper that **no** linear model on unigram counts can classify all 8 training sentences correctly.
2. Verify it in code: fit `LogisticRegression(C=100)` on `CountVectorizer()` features and on
`CountVectorizer(ngram_range=(1, 2))` features, and store both **training** accuracies.

In [ ]:
sent_texts = ["good movie", "great film", "not good", "not great",
              "bad movie", "not bad", "terrible film", "not terrible"]
sent_y = np.array([1, 1, 0, 0, 0, 1, 0, 1])
acc_unigram = None
acc_bigram = None

check('unigrams cannot fit', None if acc_unigram is None else acc_unigram < 1, True)
check('bigrams fit perfectly', acc_bigram, 1.0)

<details>
<summary><b>💡 Hint 1</b></summary>

Write the four inequalities $w_{good}+w_{movie}+b>0$, $w_{not}+w_{good}+b<0$, $w_{bad}+w_{movie}+b<0$, $w_{not}+w_{bad}+b>0$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Subtract them pairwise: what do they say about $w_{good}-w_{bad}$?

</details>

<details>
<summary><b>✅ Solution</b></summary>

From the 1st and 3rd inequalities: $w_{good} - w_{bad} > 0$. From the 2nd and 4th: $w_{bad} - w_{good} > 0$. Contradiction, so
the unigram data is not linearly separable and training accuracy is below 1. Bigram features add "not good", "not bad"
etc., which make the data separable. This is the motivation for n-grams in TF-IDF + [[Logistic Regression]] baselines,
and why contextual models (which see word order) dominate sentiment analysis.

```python
sent_texts = ["good movie", "great film", "not good", "not great",
              "bad movie", "not bad", "terrible film", "not terrible"]
sent_y = np.array([1, 1, 0, 0, 0, 1, 0, 1])
accs = []
for ngr in [(1, 1), (1, 2)]:
    Xs = CountVectorizer(ngram_range=ngr).fit_transform(sent_texts)
    clf = LogisticRegression(C=100, max_iter=2000).fit(Xs, sent_y)
    accs.append(clf.score(Xs, sent_y))
acc_unigram, acc_bigram = accs
print(acc_unigram, acc_bigram)
```

</details>

### Exercise 17 · Sampling text from your bigram LM
*💻 Code · ★★☆*

Write `generate(model, rng, max_len=10)` that starts from `<s>` and repeatedly samples the next token from the MLE bigram
distribution until `</s>`. Generate 2000 sentences and estimate how often "<s> you like dogs </s>" appears.
It should match your Part B probability.

In [ ]:
def generate(model, rng, max_len=10):
    # TODO: return the generated sentence as a string, including <s> and </s>
    return None

gen_rng = np.random.default_rng(1)
freq_you_like_dogs = None  # fraction of 2000 samples equal to "<s> you like dogs </s>"

check('Monte-Carlo frequency ≈ 1/9', freq_you_like_dogs, 1/9, tol=0.03)

<details>
<summary><b>💡 Hint</b></summary>

For context `a`, collect the candidate next tokens `b` with counts `big[(a, b)]` and sample with `rng.choice(cands, p=counts/sum)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The empirical frequency is about $0.11 \approx 1/9$, confirming that the LM is a proper distribution over sentences that we
can sample from. This is exactly how GPT generates text, just with a neural $p(x_t\mid x_{<t})$ instead of a count table;
see the decoding exercises in the LLMs notebook ([[LLMs Overview]]).

```python
def generate(model, rng, max_len=10):
    big, ctx, V = model
    toks = ['<s>']
    while toks[-1] != '</s>' and len(toks) < max_len:
        cands = [b for (a, b) in big if a == toks[-1]]
        cnt = np.array([big[(toks[-1], b)] for b in cands], dtype=float)
        toks.append(cands[rng.choice(len(cands), p=cnt / cnt.sum())])
    return " ".join(toks)

gen_rng = np.random.default_rng(1)
lm = train_bigram(lm_corpus)
samples = [generate(lm, gen_rng) for _ in range(2000)]
freq_you_like_dogs = np.mean([s == "<s> you like dogs </s>" for s in samples])
print(Counter(samples).most_common())
```

</details>

### Exercise 18 · When are BLEU and ROUGE misleading?
*🧠 Concept · ★★☆*

Give one example each where (a) a *good* output gets a low BLEU/ROUGE score and (b) a *bad* output gets a high score.
Then explain what BERTScore and LLM-as-judge do differently, and one failure mode of LLM-as-judge.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Think of paraphrases and synonyms for (a), and of a negation or a factual swap for (b).

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Reference "the kid is happy", output "the child is glad": same meaning, almost no n-gram overlap → low score.
(b) Reference "the drug reduced mortality", output "the drug did not reduce mortality": high overlap, opposite meaning.
**BERTScore** matches tokens by cosine similarity of contextual embeddings ([[Text Representations]]), so synonyms count.
**LLM-as-judge** asks a strong LLM to grade or compare outputs against a rubric. Failure modes: position bias (prefers the first
answer), verbosity bias (prefers longer answers), self-preference for its own model family, and sensitivity to prompt wording,
so calibrate it against human judgements.

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [NLP Overview](NLP%20Overview.md).*